# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _A single line item (individual order line\)_

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
# TODO: assert the row count and the quantity total

# Flatten the orders to one row per line item
lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])

# Assert row count and quantity total
assert len(lines) == 4, f"Expected 4 rows, got {len(lines)}"
assert lines['qty'].sum() == 7, f"Expected total quantity to be 7, got {lines['qty'].sum()}"

display(lines)

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
# TODO
#Join the vendor info on
vendors_df = pd.DataFrame(vendors_json)
joined = pd.merge(lines, vendors_df, on='vendor_id', how='left', indicator=True)

# Report which vendor did not match and how many units are attached to it
unmatched = joined[joined['_merge'] == 'left_only']
unmatched_vendors = unmatched['vendor_id'].unique()
unmatched_qty = unmatched['qty'].sum()

print(f"Unmatched vendor ID(s): {list(unmatched_vendors)}")
print(f"Quantity attached to unmatched vendors: {unmatched_qty}")
display(joined)

Unmatched vendor ID(s): ['V-07']
Quantity attached to unmatched vendors: 3


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# TODO
# Total quantity per vendor name

# The unmatched vendor has vendor_id 'V-07'. Since it is not in the vendor list,
# its vendor name 'name' will appear as NaN (null) in the left join results.
# Grouping by 'name' will aggregate only the matched vendors unless we explicitly
# handle or keep the NaN values.

qty_by_vendor = joined.groupby('name', dropna=False)['qty'].sum().reset_index()
display(qty_by_vendor)

,name,qty
0,Cav Merch,1
1,Hoos Burgers,3
2,NaN,3


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
# TODO
# Which zone sold the most items?
zone_sales = joined.groupby('zone', dropna=False)['qty'].sum().reset_index()
print("Sales by zone:")
display(zone_sales)

# The total quantity accounted for in defined zones is 4 units (all in Zone A).
# The remaining difference of 3 units went to the unmatched vendor ('V-07'),
# which has a zone value of NaN (null) because it was not in our vendor database.

Sales by zone:


,zone,qty
0,A,4
1,NaN,3


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
ragged_flat = []

for order in ragged_json:
    lines_list = order.get('lines', [])
    if not lines_list:
        ragged_flat.append({
            'order': order['order'],
            'vendor_id': order['vendor_id'],
            'item': None,
            'qty': None
        })
    else:
        for line in lines_list:
            ragged_flat.append({
                'order': order['order'],
                'vendor_id': order['vendor_id'],
                'item': line.get('item'),
                'qty': line.get('qty')
            })

ragged_df = pd.DataFrame(ragged_flat)
display(ragged_df)

,order,vendor_id,item,qty
0,4,V-01,Soda,2.0
1,5,V-10,None,NaN
2,6,V-01,Fries,NaN


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

**a)**

Having "one row per line item" makes sure that the dataset's granularity matches the finest level of detail present in our transactions. This shape makes it extremely easy to answer questions regarding product-specific statistics, such as what the total quantity sold per individual item is or which specific items are most frequently ordered together.

If the data remained structured at "one row per order," each row would contain nested lists or aggregated values. Answering these questions would require costly iterative unpacking every time you want to inspect details inside the order lines.


**b)**
A quantity of zero represents a known transaction state. For example, an item was explicitly selected but has a quantity of 0 (possibly due to a promotion, cancellation, or deletion). A missing quantity (`None` or `NaN`) means the information is completely unknown, not recorded, or lost.

If you conflate them (for example, by using `.fillna(0)`):
  * Descriptive statistics like the mean quantity or median will be falsely dragged down by treating missing values as actual zero-count purchases.
  * Data audits and quality flags are bypassed because you lose the ability to detect and flag systemic API or payload collection bugs. Thus, you cannot easily tell the difference between if vendor failed to record a quantity or if the customer is actually ordering zero items.